In [5]:
import sys; sys.path.append('../..'); sys.path.insert(0, '../../src/dados')
import joblib
from src.modelos.dados import ler_base, preparar, dividir
from src.modelos.variaveis import criar_variaveis
from src.modelos.incerteza import treinar_bootstrap
from src.modelos.prever import salvar_pacote

pac = joblib.load('../../modelos_salvos/pacote_prever_v1.joblib')
treino, _ = dividir(preparar(ler_base()))
treino = criar_variaveis(treino, referencia=pac['referencia'])

params = {k: v for k, v in pac['modelo'].params.items() if k != 'n_estimators'}
n_arvores = pac['modelo'].xgb.n_estimators
print(params, n_arvores)

pac['bootstrap'] = treinar_bootstrap(treino, params, n_arvores, n_reamostras=20)
salvar_pacote(pac)
print(len(pac['bootstrap']))

{'max_depth': 3, 'min_child_weight': 20, 'reg_lambda': 1} 231
20


In [6]:
from src.modelos.treino import comparar_com_bondora, _para_modelo
_, teste = dividir(preparar(ler_base()))
teste = criar_variaveis(teste, referencia=pac['referencia'])
print(comparar_com_bondora(teste, pac['modelo'].predict_proba(_para_modelo(teste))[:, 1]))

                               contratos  default_real  default_previsto  \
nosso_modelo                      7311.0        0.6708            0.5656   
pd_bondora                        7311.0        0.6708            0.3360   
nosso_modelo_mesmos_contratos     7311.0        0.6708            0.5656   

                                  auc      ks   brier  
nosso_modelo                   0.7271  0.3366  0.1987  
pd_bondora                     0.6697  0.2490  0.3226  
nosso_modelo_mesmos_contratos  0.7271  0.3366  0.1987  
